# Algorithmic Methods of Data Mining and Laboratory

# Lab Session 02 - Web Scraping

The HyperText Markup Language (HTML) is a computer language that is commonly used to define the meaning and structure of web contents through opening tags of type `<TAG>`, which denote the start of an element, and closing tags of type `</TAG>`, which denote the end of an element.

In particular, HTML syntax allows to add attributes inside the opening tag in order to provide additional information about an element.

```html
<!doctype html>
<html>
  <head>
    <title>This is the title of the webpage!</title>
  </head>
  <body>
    <p>This is an example paragraph. Anything in the <strong>body</strong> tag will appear on the page, just like this <strong>p</strong> tag and its contents.</p>
  </body>
</html>
```

It is possible to provide additional information about HTML elements by inserting some attributes within the opening tags.

For instance, the anchor element `<a>` can be used to turn text into a hyperlink that connects different web pages through the following attributes:
* `href`: The web address or destination URL.
* `title`: Additional information that is displayed using a tooltip GUI.
* `target`: Information about the browsing context

Furthermore, the content division `<div>` can be used to implement generic containers for layout styling.

In [ ]:
!pip3 install beautifulsoup4

In [ ]:
!pip3 install lxml

## Content Retrieval

Web scraping is performed using the `requests` libraries, which handles HTTP communication for content retrieval, and the `Beautiful Soup` package, which parses HTML documents through the `soup.prettify()` command.

However, to ensure that the target website does not block a scraping task, it is typically necessary to specify a custom `User-Agent` header in the request.

In [ ]:
from bs4 import BeautifulSoup
import requests

In [ ]:
headers = {'User-Agent': 'Mozilla/5.0 (Linux; Android 5.1.1; SM-G928X Build/LMY47X) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/47.0.2526.83 Mobile Safari/537.36'}
cnt = requests.get("https://en.wikipedia.org/wiki/Data_science", headers=headers)
soup = BeautifulSoup(cnt.content, features="lxml")

print(soup.prettify())

In particular, it is possible to use the `soup.find_all()` method to retrieve all the elements in the parse tree that feature a specific tag.

In [ ]:
# Find all the links within a page by searching for all elements with an anchor tag.
links = soup.find_all("a")
for link in links:
  full_link = link.get("href")
  print(full_link)

In [ ]:
text = "Etymology"
for element in soup.find_all():
  # Search for matching elements across any tag in the parse tree.
  if text == element.text:
    print(element.name)

In [ ]:
h2_tags = soup.find_all("h2")
for i, h2 in enumerate(h2_tags):
  if text in h2:
    print(i, h2)
    etymology_index = i

In [ ]:
# Specifying the `parent` attribute allows to retrieve the parent tag of an element.
div_tag = h2_tags[etymology_index].parent
print(div_tag.prettify())

In [ ]:
# The `find_next_sibling()` method allows to find for blocks that have the same tag and level as the current one.
next_div = div_tag.find_next_sibling("div")
print(next_div.prettify())

# Alternatively, one can use `find_next("p")` to retrieve just the text in the block.
next_p = next_div.find_next("p")
print(next_p.prettify())
print(next_p.text)

In [ ]:
first_h2 = h2_tags[etymology_index]
second_h2 = h2_tags[etymology_index + 1]

elements = []
for element in first_h2.next_elements:
  # Iterate through all the elements between the first two <h2> tags.
  if element == second_h2:
    # Terminate when the second tag is reached.
    break
  elements.append(element)
print(elements)

In [ ]:
# Divide into sections whenever the <h3> tag is found.
sections = {}
for element in elements:
  if element.name == "h3":
    sections[element.text] = []
  elif len(sections) > 0:
    # Split with respect to the <p> element.
    if element.name == "p":
      sections[list(sections.keys())[-1]].append(element) # Use element.text to take just the text.
print(sections)
print(sections.keys())

In [ ]:
# Extract all links within each section through the `href` tag.
href_per_section = {}
for section in sections:
  href_per_section[section] = []
  for element in sections[section]:
    for link in element.find_all('a'):
      href_per_section[section].append(link.get('href'))
print(href_per_section)

## Handling HTML Tables

Since manually parsing HTML tables tends to be complex, the `Pandas` library provides a more efficient implementation though the `pd.read_html()` function.

In fact, this function searches for any `<table>` tags and automatically parses the corresponding table before converting it into a dataframe.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from unicodedata import normalize

In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

header = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
    "X-Requested-With": "XMLHttpRequest"
}

r = requests.get(url, headers=header)
tables = pd.read_html(r.text)